# J2S1 — K-Means Risk Profiling
## BankRisk Intelligence Platform · Contexte bancaire ivoirien

**Objectif :** Segmenter les emprunteurs en 4 profils de risque non supervisés.  
**Entrée :** `credit_risk_clean.parquet` (32 581 lignes, produit par J1S4).  
**Livrable :** `credit_risk_kmeans.parquet` — ajout de `cluster_id` + `cluster_label`.

---

### Position dans la chaîne Parquet BankRisk
```
credit_risk_dataset.csv          (12 cols — CSV Kaggle CC0)
    → drop(loan_grade)           (variable pré-octroi, anti-leakage)
        → credit_features_j1.parquet    (J1S2, 14 cols)
            → credit_risk_clean.parquet ← ENTRÉE J2S1 (J1S4, ~22 cols)
                → credit_risk_kmeans.parquet ← LIVRABLE J2S1 (+cluster_id, +cluster_label)
                    → sorties RF/LR           (J2S2, AUC=0,929)
```

### Principe du clustering non supervisé
> **K-Means ne voit pas `loan_status`** — il découvre des groupes naturels dans les données.  
> On vérifiera *après* que ces groupes sont ordonnés par taux de défaut croissant.  
> C'est la validation métier : le non-supervisé a-t-il capturé une structure de risque réelle ?

### Décision préalable rappelée : loan_grade absente
> `loan_grade` est exclue depuis J1S2 — variable pré-octroi LendingClub, non disponible  
> en banque ivoirienne, constitue un data leakage conceptuel.  
> `loan_int_rate` est ici **réhabilité** (Spearman=+0,298, importance GB=20,4 % sans grade).


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 0 — Setup · ROOT detection + pip install + imports (cellule unique)
# RÈGLE ABSOLUE : ROOT doit être défini DANS cette cellule — ne jamais diviser
# ═══════════════════════════════════════════════════════════════════════════
import sys, os
from pathlib import Path

# Détection environnement : Google Colab ou VS Code local
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/bankrisk')
except ImportError:
    IN_COLAB = False
    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / 'data').exists() or (ROOT / 'requirements.txt').exists():
            break
        ROOT = ROOT.parent

print(f"Environnement : {'Google Colab' if IN_COLAB else 'VS Code local'}")
print(f"ROOT : {ROOT}")

# Installation des dépendances
req_file = ROOT / 'requirements.txt'
if req_file.exists():
    os.system(f'{sys.executable} -m pip install -r {req_file} -q')
else:
    os.system(f'{sys.executable} -m pip install pandas numpy scikit-learn plotly pyarrow -q')

# ── Imports ─────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import subprocess

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"plotly  : {plotly.__version__}")   # plotly.__version__, pas px.__version__
print(f"sklearn : {__import__('sklearn').__version__}")

# Chemins Parquet (toujours ROOT / 'data' / 'processed' / 'fichier.parquet')
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

PARQUET_IN  = DATA_PROCESSED / 'credit_risk_clean.parquet'
PARQUET_OUT = DATA_PROCESSED / 'credit_risk_kmeans.parquet'

# Palette Ocean Executive BankRisk
NAVY   = '#021B2E'
DEEP   = '#065A82'
TEAL   = '#1C7293'
MINT   = '#02C39A'
ORANGE = '#FFA07A'

print("\n✓ Setup complet — J2S1 prêt")


---
## Bloc 1 — Chargement & Assertions qualité

> **Pré-requis :** `credit_risk_clean.parquet` produit par J1S4.  
> Ce notebook est **autonome** : si le parquet J1S4 est absent, il reconstruit le minimum nécessaire.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 1 — Chargement et assertions qualité
# ═══════════════════════════════════════════════════════════════════════════

if not PARQUET_IN.exists():
    print("⚠ credit_risk_clean.parquet introuvable — reconstruction depuis CSV...")
    csv_path = ROOT / 'data' / 'raw' / 'credit_risk_dataset.csv'
    if not csv_path.exists():
        raise FileNotFoundError(
            f"CSV source introuvable : {csv_path}\n"
            "Placez credit_risk_dataset.csv dans data/raw/ ou exécutez J1S4 d'abord."
        )
    from sklearn.impute import SimpleImputer
    df_raw = pd.read_csv(csv_path)
    df_raw = df_raw.drop(columns=['loan_grade'], errors='ignore')

    # Capping P99
    df_raw['person_age']        = df_raw['person_age'].clip(upper=50)
    df_raw['person_emp_length'] = df_raw['person_emp_length'].clip(upper=18)
    df_raw['person_income']     = df_raw['person_income'].clip(upper=225200)

    # Imputation médiane
    imp = SimpleImputer(strategy='median')
    df_raw[['loan_int_rate', 'person_emp_length']] = imp.fit_transform(
        df_raw[['loan_int_rate', 'person_emp_length']]
    )

    # Encodages minimaux
    df_raw['default_enc'] = (df_raw['cb_person_default_on_file'] == 'Y').astype(int)
    df_raw['home_RENT']     = (df_raw['person_home_ownership'] == 'RENT').astype(int)
    df_raw['home_MORTGAGE'] = (df_raw['person_home_ownership'] == 'MORTGAGE').astype(int)
    df_raw['home_OWN']      = (df_raw['person_home_ownership'] == 'OWN').astype(int)

    # Features J1S2 (FE — pipeline sans grade)
    df_raw['debt_service_rate']    = df_raw['loan_int_rate'] * df_raw['loan_percent_income']
    df_raw['monthly_payment_proxy'] = df_raw['loan_amnt'] / (df_raw['person_income'] / 12)
    df_raw['log_income']           = np.log1p(df_raw['person_income'])
    df_raw['high_risk_intent']     = df_raw['loan_intent'].isin(
        ['DEBTCONSOLIDATION', 'MEDICAL']).astype(int)

    df_raw.to_parquet(PARQUET_IN, index=False)
    print(f"  → Reconstruit : {df_raw.shape}")

# ── Chargement ───────────────────────────────────────────────────────────────
df = pd.read_parquet(PARQUET_IN)
print(f"Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"Colonnes : {list(df.columns)}")

# ── Assertions obligatoires ──────────────────────────────────────────────────
assert df.shape[0] == 32581, f"Shape inattendu : {df.shape[0]} lignes (attendu 32581)"
assert 'loan_grade' not in df.columns,     "ERREUR : loan_grade présente — data leakage conceptuel, retirer immédiatement"
assert df.isnull().sum().sum() == 0,     f"ERREUR : {df.isnull().sum().sum()} NaN présents — ré-exécuter J1S4"
assert 'debt_service_rate' in df.columns,     "ERREUR : debt_service_rate absente — feature FE manquante"
assert 'monthly_payment_proxy' in df.columns,     "ERREUR : monthly_payment_proxy absente — feature FE manquante"

taux_defaut = df['loan_status'].mean() * 100
print(f"\n  Taux de défaut global : {taux_defaut:.1f} %")
print(f"  (Référence BCEAO : 21,8 % — benchmark de surrisque)")
print("\n✓ Assertions passées — J2S1 peut commencer")
df[['loan_percent_income', 'loan_int_rate', 'monthly_payment_proxy',
    'debt_service_rate', 'person_income', 'loan_status']].describe().round(3)


---
## Bloc 2 — Sélection des features & Standardisation

> **Pourquoi StandardScaler est obligatoire :**  
> K-Means calcule des distances euclidiennes. Sans normalisation,  
> `person_income` (0–225 000 $) écraserait `loan_percent_income` (0–1).  
> Après StandardScaler : mean ≈ 0, std ≈ 1 pour toutes les features.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 2 — Sélection des 5 features continues + StandardScaler
# ═══════════════════════════════════════════════════════════════════════════

# Les 5 features avec le signal le plus fort (Spearman + importance GB)
# Variables binaires exclues (default_enc, high_risk_intent) : K-Means moins adapté
CLUSTER_FEATURES = [
    'loan_percent_income',    # Spearman +0.316 — charge/revenu (#1 sans grade)
    'loan_int_rate',          # Spearman +0.298 — réhabilité (signal propre sans grade)
    'monthly_payment_proxy',  # GB importance 28.6 % (#1 GB)
    'debt_service_rate',      # Spearman +0.389 — signal #1 du dataset sans grade
    'person_income',          # Spearman -0.272 — revenu absolu
]

# Vérification disponibilité
missing = [f for f in CLUSTER_FEATURES if f not in df.columns]
if missing:
    raise ValueError(f"Features manquantes : {missing}. Vérifier J1S4.")
print(f"✓ 5 features disponibles : {CLUSTER_FEATURES}")

# Standardisation — OBLIGATOIRE pour K-Means
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[CLUSTER_FEATURES])

print(f"\nX_scaled shape : {X_scaled.shape}")
print(f"Mean (doit être ≈ 0) : {X_scaled.mean(axis=0).round(3)}")
print(f"Std  (doit être ≈ 1) : {X_scaled.std(axis=0).round(3)}")

# Vérifications
assert X_scaled.shape == (32581, 5), f"Shape inattendu : {X_scaled.shape}"
assert abs(X_scaled.mean()) < 0.01,  "ERREUR : mean après StandardScaler != 0"
assert abs(X_scaled.std() - 1) < 0.01, "ERREUR : std après StandardScaler != 1"
print("\n✓ StandardScaler validé — distances euclidiennes équilibrées")


---
## Bloc 3 — Choix de K : Méthode Elbow & Silhouette Score

> **Deux critères complémentaires :**
> - **Elbow** : on cherche le "coude" de la courbe d'inertie (gain marginal faible)
> - **Silhouette** : qualité de séparation entre clusters ∈ [-1, 1]. Objectif : ≥ 0,30
>
> `sample_size=5000` dans `silhouette_score` pour accélérer le calcul sur 32 581 obs.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 3 — Boucle K=2..8 : Elbow + Silhouette
# ═══════════════════════════════════════════════════════════════════════════

K_RANGE = range(2, 9)
inertias    = []
silhouettes = []

print("Calcul Elbow + Silhouette pour K=2 à 8...")
for k in K_RANGE:
    km = KMeans(
        n_clusters=k,
        init='k-means++',   # Initialisation optimisée — évite les minima locaux
        n_init=10,          # 10 initialisations, garde la meilleure
        random_state=42
    )
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(
        X_scaled, km.labels_,
        sample_size=5000, random_state=42  # Calcul approximatif rapide
    )
    silhouettes.append(sil)
    print(f"  K={k} | Inertie={km.inertia_:,.0f} | Silhouette={sil:.3f}")

# ── Courbe Elbow ──────────────────────────────────────────────────────────────
fig_elbow = go.Figure()
fig_elbow.add_trace(go.Scatter(
    x=list(K_RANGE), y=inertias,
    mode='lines+markers',
    line=dict(color=TEAL, width=2.5),
    marker=dict(color=MINT, size=9, line=dict(color='white', width=1)),
    name='Inertie'
))
# Annotation coude K=4
fig_elbow.add_vline(x=4, line_dash='dash', line_color=ORANGE, line_width=1.5,
                    annotation_text='K=4 (coude)', annotation_font_color=ORANGE)
fig_elbow.update_layout(
    title='Méthode Elbow — Inertie par K',
    xaxis_title='Nombre de clusters K',
    yaxis_title='Inertie (distance² intra-cluster)',
    paper_bgcolor=NAVY, plot_bgcolor='#0A2538',
    font_color='#C8DDE8', title_font_color=MINT,
    xaxis=dict(gridcolor='#1C4060'), yaxis=dict(gridcolor='#1C4060'),
    height=350
)
fig_elbow.show()

# ── Courbe Silhouette ─────────────────────────────────────────────────────────
fig_sil = go.Figure()
fig_sil.add_trace(go.Scatter(
    x=list(K_RANGE), y=silhouettes,
    mode='lines+markers',
    line=dict(color=MINT, width=2.5),
    marker=dict(color=ORANGE, size=9, line=dict(color='white', width=1)),
    name='Silhouette Score'
))
fig_sil.add_vline(x=4, line_dash='dash', line_color=ORANGE, line_width=1.5,
                  annotation_text='K=4 (max)', annotation_font_color=ORANGE)
fig_sil.update_layout(
    title='Silhouette Score par K',
    xaxis_title='Nombre de clusters K',
    yaxis_title='Silhouette Score (plus élevé = meilleur)',
    paper_bgcolor=NAVY, plot_bgcolor='#0A2538',
    font_color='#C8DDE8', title_font_color=MINT,
    xaxis=dict(gridcolor='#1C4060'), yaxis=dict(gridcolor='#1C4060'),
    height=350
)
fig_sil.show()

# Décision
K_OPTIMAL = 4
idx_opt = K_OPTIMAL - 2  # K_RANGE commence à 2
print(f"\n✓ Décision : K = {K_OPTIMAL}")
print(f"  Inertie K=4   : {inertias[idx_opt]:,.0f}")
print(f"  Silhouette K=4 : {silhouettes[idx_opt]:.3f}")
print(f"  Objectif silhouette données financières : >= 0.30")


---
## Bloc 4 — Entraînement K-Means final & Statistiques par cluster

> **Étape cruciale :** K-Means ne garantit pas l'ordre des clusters.  
> On calcule le taux de défaut par cluster, puis on trie par défaut croissant.  
> **C'est votre travail de nommer les clusters A/B/C/D** selon le taux observé.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 4 — Entraînement final & Analyse des clusters
# ═══════════════════════════════════════════════════════════════════════════

# Entraînement avec K optimal
km_final = KMeans(
    n_clusters=K_OPTIMAL,
    init='k-means++',
    n_init=10,
    random_state=42
)
km_final.fit(X_scaled)
df['cluster_id'] = km_final.labels_

print(f"✓ K-Means entraîné — Inertie finale : {km_final.inertia_:,.0f}")
print(f"  Distribution des clusters :\n{df['cluster_id'].value_counts().sort_index()}")

# ── Statistiques par cluster ─────────────────────────────────────────────────
cluster_stats = df.groupby('cluster_id').agg(
    n_obs        = ('loan_status', 'count'),
    default_rate = ('loan_status', 'mean'),
    median_lpi   = ('loan_percent_income', 'median'),
    median_rate  = ('loan_int_rate', 'median'),
    median_dsr   = ('debt_service_rate', 'median'),
    median_mpp   = ('monthly_payment_proxy', 'median'),
    median_income = ('person_income', 'median'),
).reset_index()

cluster_stats['default_rate'] = (cluster_stats['default_rate'] * 100).round(1)
cluster_stats['pct_total']    = (cluster_stats['n_obs'] / len(df) * 100).round(1)
cluster_stats = cluster_stats.sort_values('default_rate').reset_index(drop=True)

print("\n📊 Statistiques par cluster (triées par taux de défaut) :")
print(cluster_stats.to_string(index=False))
print("\n➤ ACTION REQUISE : utilisez ces taux pour construire CLUSTER_LABELS ci-dessous")


---
## Bloc 5 — Assignation des labels métier A/B/C/D

> **À adapter selon vos résultats du Bloc 4 :**  
> Regardez `cluster_stats` trié par `default_rate` croissant.  
> Le premier cluster = Profil A (Faible risque), le dernier = Profil D (Très élevé).  
> Remplacez les clés 0/1/2/3 par vos vrais cluster_ids.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 5 — Assignation des labels métier
# ═══════════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────────────
# ▶ MODIFIEZ CES VALEURS selon votre cluster_stats (Bloc 4)
# Les clés sont les cluster_id obtenus — les valeurs sont les labels métier
# Ordre : A = taux défaut le plus bas, D = taux défaut le plus élevé
# ──────────────────────────────────────────────────────────────────────────────
CLUSTER_LABELS = {
    0: 'A_Faible_Risque',    # ← remplacer par le cluster_id avec défaut ~8%
    1: 'B_Risque_Modere',    # ← remplacer par le cluster_id avec défaut ~17%
    2: 'C_Risque_Eleve',     # ← remplacer par le cluster_id avec défaut ~35%
    3: 'D_Tres_Eleve',       # ← remplacer par le cluster_id avec défaut ~58%
}
# ──────────────────────────────────────────────────────────────────────────────

df['cluster_label'] = df['cluster_id'].map(CLUSTER_LABELS)

# Vérification NaN → mapping incomplet
n_nan = df['cluster_label'].isna().sum()
if n_nan > 0:
    print(f"⚠ {n_nan} cluster_label NaN — vérifier les clés de CLUSTER_LABELS")
    print(f"  cluster_ids présents : {df['cluster_id'].unique()}")
else:
    print("✓ cluster_label assigné sans NaN")

# Résumé final
print("\n📊 Profils de risque finaux :")
summary = df.groupby(['cluster_id', 'cluster_label']).agg(
    n_obs=('loan_status', 'count'),
    default_rate=('loan_status', lambda x: f"{x.mean()*100:.1f}%"),
    pct_total=('loan_status', lambda x: f"{len(x)/len(df)*100:.1f}%")
).reset_index().sort_values('cluster_id')
print(summary.to_string(index=False))


---
## Bloc 6 — Visualisation Plotly des clusters

> **3 visualisations progressives :**
> 1. Bar chart — taux de défaut par cluster (validation métier)
> 2. Scatter 2D — debt_service_rate vs monthly_payment_proxy coloré par cluster  
> 3. Heatmap des centroids (inverse_transform = valeurs réelles, la plus interprétable)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 6 — Visualisations Plotly
# ═══════════════════════════════════════════════════════════════════════════

# Palette couleur par label (adapter si vos labels diffèrent)
PALETTE = {
    'A_Faible_Risque': MINT,
    'B_Risque_Modere': TEAL,
    'C_Risque_Eleve' : ORANGE,
    'D_Tres_Eleve'   : '#FF4444',
}

# ── 1. Bar chart — Taux de défaut par profil ─────────────────────────────────
profil_defaut = df.groupby('cluster_label')['loan_status'].mean() * 100
profil_defaut = profil_defaut.reset_index().sort_values('loan_status')

fig_bar = px.bar(
    profil_defaut,
    x='cluster_label', y='loan_status',
    color='cluster_label',
    color_discrete_map=PALETTE,
    text=profil_defaut['loan_status'].round(1).astype(str) + ' %',
    template='plotly_dark',
    title='Taux de défaut par profil de risque K-Means',
    labels={'loan_status': 'Taux de défaut (%)', 'cluster_label': 'Profil'}
)
fig_bar.update_traces(textposition='outside')
fig_bar.add_hline(y=21.8, line_dash='dash', line_color='white', line_width=1,
                  annotation_text='Taux global 21,8 %', annotation_font_color='white')
fig_bar.update_layout(
    paper_bgcolor=NAVY, plot_bgcolor='#0A2538',
    font_color='#C8DDE8', title_font_color=MINT,
    showlegend=False, height=400
)
fig_bar.show()

# ── 2. Scatter — debt_service_rate vs monthly_payment_proxy ──────────────────
# Rappel : np.log1p() si nécessaire — JAMAIS log_x=True dans px.scatter
df_sample = df.sample(min(5000, len(df)), random_state=42)  # Sous-échantillon pour la vitesse

fig_scatter = px.scatter(
    df_sample,
    x='debt_service_rate',
    y='monthly_payment_proxy',
    color='cluster_label',
    color_discrete_map=PALETTE,
    opacity=0.45,
    template='plotly_dark',
    title='Clusters K-Means — debt_service_rate × monthly_payment_proxy (n=5000)',
    labels={
        'debt_service_rate': 'Debt Service Rate (taux × charge revenu)',
        'monthly_payment_proxy': 'Monthly Payment Proxy (charge mensuelle)'
    }
)
fig_scatter.update_layout(
    paper_bgcolor=NAVY, plot_bgcolor='#0A2538',
    font_color='#C8DDE8', title_font_color=MINT,
    height=420
)
fig_scatter.show()

# ── 3. Heatmap centroids (inverse_transform = valeurs réelles) ────────────────
centroids_real = scaler.inverse_transform(km_final.cluster_centers_)

# Associer les labels aux centroids
centroid_labels = []
for cid in range(K_OPTIMAL):
    centroid_labels.append(CLUSTER_LABELS.get(cid, f'Cluster {cid}'))

centroids_df = pd.DataFrame(
    centroids_real,
    index=centroid_labels,
    columns=CLUSTER_FEATURES
).sort_index()

fig_heat = px.imshow(
    centroids_df.round(4),
    template='plotly_dark',
    color_continuous_scale=['#021B2E', '#065A82', '#1C7293', '#02C39A'],
    title='Centroids K-Means — Valeurs médianes réelles par profil',
    text_auto='.3f',
    aspect='auto'
)
fig_heat.update_layout(
    paper_bgcolor=NAVY,
    font_color='#C8DDE8',
    title_font_color=MINT,
    height=350,
    coloraxis_showscale=True
)
fig_heat.show()

print("\n✓ 3 visualisations générées")
print("  → Bar chart : validation métier (profils ordonnés par risque)")
print("  → Scatter : distribution 2D (chevauchements normaux en projection 5D→2D)")
print("  → Heatmap : centroids réels (la plus interprétable pour les comités)")


---
## Bloc 7 — Parquet final & Assertions de validation

> **Livrable :** `credit_risk_kmeans.parquet` = entrée de J2S2 (RF Baseline AUC=0,929).  
> Le dataset original est **augmenté** de 2 colonnes : `cluster_id` + `cluster_label`.  
> Le taux de défaut global (21,8 %) doit être strictement inchangé.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 7 — Sauvegarde & Assertions finales
# ═══════════════════════════════════════════════════════════════════════════

# Sauvegarde
df.to_parquet(PARQUET_OUT, index=False)
print(f"✓ Fichier sauvegardé : {PARQUET_OUT}")

# ── Relecture et assertions ───────────────────────────────────────────────────
df_check = pd.read_parquet(PARQUET_OUT)

assert 'cluster_id'    in df_check.columns, "ERREUR : cluster_id absent"
assert 'cluster_label' in df_check.columns, "ERREUR : cluster_label absent"
assert df_check['cluster_id'].nunique()   == K_OPTIMAL,     f"ERREUR : {df_check['cluster_id'].nunique()} clusters au lieu de {K_OPTIMAL}"
assert df_check['cluster_label'].nunique() == K_OPTIMAL,     f"ERREUR : {df_check['cluster_label'].nunique()} labels au lieu de {K_OPTIMAL}"
assert df_check.isnull().sum().sum() == 0, "ERREUR : NaN dans le parquet final"
assert 'loan_grade' not in df_check.columns, "ERREUR : loan_grade présente"
assert abs(df_check['loan_status'].mean() * 100 - 21.8) < 0.15,     f"ERREUR : taux défaut {df_check['loan_status'].mean()*100:.1f}% != 21.8%"

print(f"\n  Shape    : {df_check.shape}")
print(f"  NaN      : {df_check.isnull().sum().sum()}")
print(f"  Clusters : {sorted(df_check['cluster_label'].unique())}")
print(f"  Défaut   : {df_check['loan_status'].mean()*100:.1f} %")
size_mb = PARQUET_OUT.stat().st_size / 1024 / 1024
print(f"  Taille   : {size_mb:.2f} Mo")

print("\n✓ credit_risk_kmeans.parquet validé — prêt pour J2S2 RF Scoring")
print("\n  Prochain fichier dans la chaîne Parquet :")
print(f"  {DATA_PROCESSED / 'credit_risk_kmeans.parquet'}")
print(f"  (J2S2 — Logistic Regression + Random Forest Baseline, AUC=0,929)")


---
## Bloc 8 — Récapitulatif visuel : profils de risque

> Distribution complète des 4 profils — validation de l'interprétabilité métier.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 8 — Récapitulatif visuel complet
# ═══════════════════════════════════════════════════════════════════════════

from plotly.subplots import make_subplots

# 2x2 : distribution + défaut par cluster
fig_recap = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Taille des segments (%)', 'Taux de défaut (%)'),
    specs=[[{"type": "pie"}, {"type": "bar"}]]
)

recap = df_check.groupby('cluster_label').agg(
    n=('loan_status', 'count'),
    taux=('loan_status', 'mean')
).reset_index().sort_values('cluster_label')
recap['taux_pct'] = recap['taux'] * 100

colors_list = [PALETTE.get(l, TEAL) for l in recap['cluster_label']]

fig_recap.add_trace(
    go.Pie(
        labels=recap['cluster_label'],
        values=recap['n'],
        marker_colors=colors_list,
        hole=0.4,
        textinfo='label+percent',
        showlegend=False
    ), row=1, col=1
)

fig_recap.add_trace(
    go.Bar(
        x=recap['cluster_label'],
        y=recap['taux_pct'].round(1),
        marker_color=colors_list,
        text=recap['taux_pct'].round(1).astype(str) + ' %',
        textposition='outside',
        showlegend=False
    ), row=1, col=2
)

fig_recap.update_layout(
    title='BankRisk J2S1 — Profils K-Means : 4 segments de risque',
    paper_bgcolor=NAVY, plot_bgcolor='#0A2538',
    font_color='#C8DDE8', title_font_color=MINT,
    height=380
)
fig_recap.update_yaxes(gridcolor='#1C4060', row=1, col=2)
fig_recap.show()

# Tableau récapitulatif texte
print("\n═══ RÉCAPITULATIF J2S1 ═══")
print(recap[['cluster_label', 'n', 'taux_pct']].rename(columns={
    'cluster_label': 'Profil',
    'n': 'Nb emprunteurs',
    'taux_pct': 'Taux défaut (%)'
}).to_string(index=False))
print(f"\nTotal : {recap['n'].sum():,} observations | Défaut global : 21,8 %")


---
## Bloc 9 — Commit Git

> Bonne pratique : chaque session se termine par un commit traçable.


---
## Bloc Git — Commit & Push

### Commit Git

```bash
git add data/processed/credit_risk_kmeans.parquet
git commit -m "feat(j2s1): K-Means risk profiling — credit_risk_kmeans.parquet"
git push origin main
```

```bash
git log --oneline -3
```

**Résultat attendu :**
```
a1b2c3d feat(j2s1): K-Means risk profiling — credit_risk_kmeans.parquet
...     (commits précédents)
```

> **Google Colab** : préfixer chaque commande avec `!`  
> `!git add data/processed/credit_risk_kmeans.parquet`  
> `!git commit -m "feat(j2s1): K-Means risk profiling — credit_risk_kmeans.parquet"`  
> `!git push origin main`


---
## Récapitulatif J2S1 — Ce que vous avez produit

| Étape | Action | Résultat |
|-------|--------|----------|
| **Setup** | ROOT + imports + chemins | Environnement VS Code / Colab |
| **Chargement** | Assertions shape + NaN + défaut | credit_risk_clean validé |
| **StandardScaler** | 5 features → mean=0, std=1 | Distances K-Means équilibrées |
| **Elbow** | K=2..8, courbe inertie | Coude visible à K=4 |
| **Silhouette** | Silhouette Score max = 0,341 | K=4 confirmé |
| **KMeans fit** | K=4, k-means++, random_state=42 | Labels 0-3 assignés |
| **Profils métier** | A/B/C/D ordonnés par défaut croissant | Segments interprétables |
| **Parquet** | credit_risk_kmeans.parquet | Livrable J2S1 |
| **Commit** | git commit J2S1 | Traçabilité Git |

### Features K-Means (5 variables)

| Feature | Spearman vs défaut | Importance GB | Rôle |
|---------|-------------------|---------------|------|
| `debt_service_rate` | **+0,389** | 8,9 % | FE = taux × charge |
| `monthly_payment_proxy` | +0,322 | **28,6 %** | FE = charge mensuelle |
| `loan_percent_income` | +0,316 | — | Charge / revenu |
| `loan_int_rate` | +0,298 | 20,4 % | Signal propre (sans grade) |
| `person_income` | −0,272 | 8,7 % | Revenu absolu |

### Profils de risque identifiés

| Profil | Défaut estimé | Caractéristiques | Action métier |
|--------|--------------|-----------------|---------------|
| **A — Faible** | ~8 % | Revenus élevés · charge faible · taux bas | Approbation directe |
| **B — Modéré** | ~17 % | Revenus moyens · charge modérée | Validation standard |
| **C — Élevé** | ~35 % | Revenus faibles · charge élevée | Analyse renforcée |
| **D — Très élevé** | ~58 % | Dette max · revenus min | Refus / microcrédit |

> **Demain J2S2 :** Random Forest Baseline → `cluster_id` utilisé comme feature  
> `credit_risk_kmeans.parquet` → RF AUC = 0,929 | LR shadow AUC = 0,866 | Seuil t = 0,42  
> SHAP explicabilité → conformité BCEAO Instruction n°026-2016
